# International Debt — SQL Analysis
**Author:** Nobukhosi Sibanda
**Data Source:** World Bank International Debt Statistics (IDS), 2024
**Focus:** GROUP BY aggregation, running totals, and comparing values across countries, indicators, and regions in SQL.

This notebook builds a SQLite database from live World Bank data, then works through the SQL queries in
[`sql/analysis.sql`](../sql/analysis.sql) to answer:

- How much external debt does each country carry, and who owes the most / least in absolute terms?
- What does the average debt burden look like, and which debt indicators are most commonly reported?
- How does debt compare across regions, income levels, and relative to each country's own GDP/GNI?

The qualitative "why" behind the top and bottom 10 — economic structure, political context, and what
heavily indebted countries could do about it — is written up in the [README](../README.md#country-deep-dive-top-10-most-indebted)
after the numbers below establish *what* the data shows. The same figures also drive the
[interactive dashboard](../assets/international_debt_dashboard.html).


In [1]:
import sqlite3
import pandas as pd
import plotly.graph_objects as go

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

conn = sqlite3.connect('../international_debt.db')


## 1. Data overview

Two tables, built by `src/fetch_data.py` straight from the World Bank Open Data API (`api.worldbank.org`):

- **`international_debt`** — long format: one row per (country, debt indicator), e.g. "Brazil / External debt stocks, total / $605.5B".
- **`country_reference`** — one row per country: region, income level, GDP, population, and debt-to-GNI / debt-service ratios.


In [2]:
pd.read_sql_query("SELECT * FROM international_debt LIMIT 5", conn)

,country_name,country_code,indicator_name,indicator_code,debt
0,Afghanistan,AFG,"External debt stocks, total (DOD, current US$)",DT.DOD.DECT.CD,"3,344,095,867.10"
1,Albania,ALB,"External debt stocks, total (DOD, current US$)",DT.DOD.DECT.CD,"10,712,630,800.80"
2,Algeria,DZA,"External debt stocks, total (DOD, current US$)",DT.DOD.DECT.CD,"6,898,271,361.00"
3,Angola,AGO,"External debt stocks, total (DOD, current US$)",DT.DOD.DECT.CD,"58,733,063,313.60"
4,Argentina,ARG,"External debt stocks, total (DOD, current US$)",DT.DOD.DECT.CD,"242,357,155,703.10"


In [3]:
pd.read_sql_query("SELECT * FROM country_reference LIMIT 5", conn)

,country_code,country_name,region,income_level,DT.DOD.DECT.GN.ZS,DT.DOD.DSTC.ZS,DT.TDS.DECT.EX.ZS,gdp_current_usd,gdp_per_capita_usd,population,total_reserves_usd
0,ABW,Aruba,Latin America & Caribbean,High income,NaN,NaN,NaN,"4,167,588,070.29","38,590.57",107995,"1,902,250,220.00"
1,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income,NaN,13.24,NaN,"17,778,508,875.74",416.87,42647492,NaN
2,AGO,Angola,Sub-Saharan Africa,Lower middle income,79.57,9.89,28.91,"103,080,538,044.06","2,720.82",37885849,"14,242,892,504.62"
3,ALB,Albania,Europe & Central Asia,Upper middle income,39.72,11.58,8.01,"27,037,474,263.30","11,374.01",2377128,"6,515,581,527.60"
4,AND,Andorra,Europe & Central Asia,High income,NaN,NaN,NaN,"4,044,249,678.66","49,357.44",81938,NaN


## 2. Grouping & totals

The core SQL skill this project is built around: `GROUP BY` to roll many rows into one summary row per
country / indicator / region, and `SUM` / `AVG` / `COUNT` to turn raw debt figures into totals worth
comparing.


In [4]:
query = """
SELECT
    COUNT(DISTINCT country_name) AS distinct_countries,
    COUNT(DISTINCT indicator_code) AS distinct_indicators,
    COUNT(*) AS total_rows
FROM international_debt;
"""
pd.read_sql_query(query, conn)

,distinct_countries,distinct_indicators,total_rows
0,120,10,1082


In [5]:
query = """
SELECT ROUND(SUM(debt) / 1e12, 2) AS total_external_debt_trillions_usd
FROM international_debt
WHERE indicator_code = 'DT.DOD.DECT.CD';
"""
pd.read_sql_query(query, conn)

,total_external_debt_trillions_usd
0,8.94


In [6]:
query = """
SELECT indicator_name, indicator_code,
       COUNT(*) AS countries_reporting,
       ROUND(AVG(debt) / 1e9, 2) AS avg_debt_billions_usd
FROM international_debt
GROUP BY indicator_name, indicator_code
ORDER BY avg_debt_billions_usd DESC;
"""
avg_by_indicator = pd.read_sql_query(query, conn)
avg_by_indicator

,indicator_name,indicator_code,countries_reporting,avg_debt_billions_usd
0,"External debt stocks, total (DOD, current US$)",DT.DOD.DECT.CD,118,75.72
1,"External debt stocks, public and publicly guar...",DT.DOD.DPPG.CD,118,30.16
2,"External debt stocks, private nonguaranteed (P...",DT.DOD.DPNG.CD,92,28.39
3,"External debt stocks, short-term (DOD, current...",DT.DOD.DSTC.CD,118,20.29
4,"Debt service on external debt, total (TDS, cur...",DT.TDS.DECT.CD,120,11.31
5,"Debt service on external debt, public and publ...",DT.TDS.DPPG.CD,120,4.11
6,"PPG, IBRD (DOD, current US$)",DT.DOD.MIBR.CD,63,4.03
7,"IBRD loans and IDA credits (DOD, current US$)",DT.DOD.MWBG.CD,116,3.99
8,"PPG, IDA (DOD, current US$)",DT.DOD.MIDA.CD,97,2.16
9,"Multilateral debt service (TDS, current US$)",DT.TDS.MLAT.CD,120,0.85


**Most commonly reported indicator** — i.e. which debt metric has the best country coverage:

In [7]:
query = """
SELECT indicator_name, COUNT(DISTINCT country_name) AS countries_reporting
FROM international_debt
GROUP BY indicator_name
ORDER BY countries_reporting DESC;
"""
pd.read_sql_query(query, conn)

,indicator_name,countries_reporting
0,"Multilateral debt service (TDS, current US$)",120
1,"Debt service on external debt, total (TDS, cur...",120
2,"Debt service on external debt, public and publ...",120
3,"External debt stocks, total (DOD, current US$)",118
4,"External debt stocks, short-term (DOD, current...",118
5,"External debt stocks, public and publicly guar...",118
6,"IBRD loans and IDA credits (DOD, current US$)",116
7,"PPG, IDA (DOD, current US$)",97
8,"External debt stocks, private nonguaranteed (P...",92
9,"PPG, IBRD (DOD, current US$)",63


## 3. Who owes the most? Who owes the least?

Ranking by the headline `DT.DOD.DECT.CD` indicator — **External debt stocks, total (current US$)**.


In [8]:
query = """
SELECT country_name, ROUND(debt / 1e9, 2) AS total_external_debt_billions_usd
FROM international_debt
WHERE indicator_code = 'DT.DOD.DECT.CD'
ORDER BY debt DESC
LIMIT 10;
"""
top10 = pd.read_sql_query(query, conn)
top10

,country_name,total_external_debt_billions_usd
0,China,"2,419.84"
1,India,716.46
2,Brazil,605.46
3,Mexico,591.26
4,Turkiye,514.99
5,Indonesia,421.06
6,Argentina,242.36
7,Colombia,201.76
8,Ukraine,193.49
9,Thailand,191.83


In [9]:
query = """
SELECT country_name, ROUND(debt / 1e9, 3) AS total_external_debt_billions_usd
FROM international_debt
WHERE indicator_code = 'DT.DOD.DECT.CD'
ORDER BY debt ASC
LIMIT 10;
"""
bottom10 = pd.read_sql_query(query, conn)
bottom10

,country_name,total_external_debt_billions_usd
0,Tonga,0.17
1,Timor-Leste,0.30
2,Sao Tome and Principe,0.33
3,Comoros,0.39
4,Samoa,0.40
5,Vanuatu,0.52
6,Dominica,0.57
7,Solomon Islands,0.60
8,Eritrea,0.69
9,St. Vincent and the Grenadines,0.81


In [10]:
fig = go.Figure()
fig.add_bar(x=top10.total_external_debt_billions_usd[::-1], y=top10.country_name[::-1],
            orientation='h', marker_color='#2a78d6', name='Top 10')
fig.update_layout(title='Top 10 Countries by Total External Debt (US$ billions)',
                   height=420, margin=dict(l=10, r=10, t=40, b=10))
fig.show()

**Reading the ranking:** the top 10 is dominated by large, industrializing middle-income
economies (China, India, Brazil, Mexico, Türkiye, Indonesia) — they carry the biggest absolute debt
loads simply because they run the biggest economies and have the deepest access to international
capital markets. The bottom 10 is almost entirely small island and micro-economies (Tonga,
Timor-Leste, São Tomé and Príncipe, Comoros, Samoa...) — a small dollar figure here mostly reflects a
tiny GDP and limited market access, **not** necessarily healthy debt management. Section 5 below
normalizes for this.


## 4. Comparing across regions and income levels

In [11]:
query = """
SELECT r.region, COUNT(*) AS countries,
       ROUND(SUM(d.debt) / 1e9, 1) AS total_debt_billions_usd,
       ROUND(AVG(d.debt) / 1e9, 2) AS avg_debt_per_country_billions_usd
FROM international_debt d
JOIN country_reference r ON d.country_code = r.country_code
WHERE d.indicator_code = 'DT.DOD.DECT.CD'
GROUP BY r.region
ORDER BY total_debt_billions_usd DESC;
"""
pd.read_sql_query(query, conn)

,region,countries,total_debt_billions_usd,avg_debt_per_country_billions_usd
0,East Asia & Pacific,16,"3,409.40",213.08
1,Latin America & Caribbean,21,"1,999.70",95.22
2,Europe & Central Asia,18,"1,169.30",64.96
3,Sub-Saharan Africa,44,899.60,20.45
4,South Asia,6,896.00,149.34
5,"Middle East, North Africa, Afghanistan & Pakistan",13,561.20,43.17


In [12]:
query = """
SELECT r.income_level, COUNT(*) AS countries,
       ROUND(SUM(d.debt) / 1e9, 1) AS total_debt_billions_usd
FROM international_debt d
JOIN country_reference r ON d.country_code = r.country_code
WHERE d.indicator_code = 'DT.DOD.DECT.CD'
GROUP BY r.income_level
ORDER BY total_debt_billions_usd DESC;
"""
pd.read_sql_query(query, conn)

,income_level,countries,total_debt_billions_usd
0,Upper middle income,51,"6,725.40"
1,Lower middle income,43,"1,968.00"
2,Low income,23,238.10
3,High income,1,3.70


## 5. Normalized comparisons — debt relative to the size of the economy

Raw dollar rankings favor big economies. Two ratios the IMF/World Bank actually use to flag debt
distress risk: debt as a share of national income (GNI), and debt service as a share of export
earnings (the hard currency available to actually pay it back).


In [13]:
query = """
SELECT country_name, ROUND("DT.DOD.DECT.GN.ZS", 1) AS external_debt_pct_of_gni
FROM country_reference
WHERE "DT.DOD.DECT.GN.ZS" IS NOT NULL
ORDER BY "DT.DOD.DECT.GN.ZS" DESC
LIMIT 10;
"""
pd.read_sql_query(query, conn)

,country_name,external_debt_pct_of_gni
0,Mozambique,350.60
1,Mongolia,181.90
2,Senegal,150.70
3,Mauritius,123.20
4,Lao PDR,115.40
5,Zambia,114.20
6,Montenegro,108.60
7,Ukraine,101.30
8,Rwanda,93.90
9,Jordan,90.10


In [14]:
query = """
SELECT country_name, ROUND("DT.TDS.DECT.EX.ZS", 1) AS debt_service_pct_of_exports
FROM country_reference
WHERE "DT.TDS.DECT.EX.ZS" IS NOT NULL
ORDER BY "DT.TDS.DECT.EX.ZS" DESC
LIMIT 10;
"""
pd.read_sql_query(query, conn)

,country_name,debt_service_pct_of_exports
0,El Salvador,96.20
1,Haiti,63.20
2,"Egypt, Arab Rep.",49.20
3,Kazakhstan,48.40
4,Mozambique,46.50
5,Papua New Guinea,43.30
6,Colombia,43.00
7,Senegal,41.80
8,Pakistan,39.50
9,Argentina,38.30


Notice this is almost a **completely different set of countries** than the top-10-by-raw-debt
list — Mozambique, Mongolia, Senegal, Mauritius are not among the biggest borrowers in dollar terms,
but they carry the heaviest burden relative to what their economies actually produce.


## 6. Debt vs. GDP — is bigger debt actually riskier?

In [15]:
query = """
SELECT d.country_name,
       ROUND(d.debt / 1e9, 1) AS total_debt_billions_usd,
       ROUND(r.gdp_current_usd / 1e9, 1) AS gdp_billions_usd,
       ROUND(100.0 * d.debt / NULLIF(r.gdp_current_usd, 0), 1) AS debt_pct_of_gdp
FROM international_debt d
JOIN country_reference r ON d.country_code = r.country_code
WHERE d.indicator_code = 'DT.DOD.DECT.CD'
ORDER BY d.debt DESC
LIMIT 20;
"""
debt_vs_gdp = pd.read_sql_query(query, conn)
debt_vs_gdp

,country_name,total_debt_billions_usd,gdp_billions_usd,debt_pct_of_gdp
0,China,"2,419.80","18,729.70",12.90
1,India,716.50,"3,760.80",19.10
2,Brazil,605.50,"2,185.80",27.70
3,Mexico,591.30,"1,830.50",32.30
4,Turkiye,515.00,"1,359.10",37.90
5,Indonesia,421.10,"1,396.30",30.20
6,Argentina,242.40,638.40,38.00
7,Colombia,201.80,420.50,48.00
8,Ukraine,193.50,190.80,101.40
9,Thailand,191.80,529.40,36.20


**Ukraine is the outlier**: the only economy in the top 20 debtors whose external debt (101.4%
of GDP) exceeds its entire annual economic output — a direct, mechanical consequence of financing
the war against Russia's full-scale invasion since February 2022 largely through external borrowing
and grants. Every other top-20 debtor sits between 13% and 58% of GDP.


## 7. Case study — how is the largest debtor's debt actually structured?

In [16]:
query = """
SELECT indicator_name, ROUND(debt / 1e9, 2) AS debt_billions_usd
FROM international_debt
WHERE country_name = (
    SELECT country_name FROM international_debt
    WHERE indicator_code = 'DT.DOD.DECT.CD'
    ORDER BY debt DESC LIMIT 1
)
ORDER BY debt_billions_usd DESC;
"""
pd.read_sql_query(query, conn)

,indicator_name,debt_billions_usd
0,"External debt stocks, total (DOD, current US$)","2,419.84"
1,"External debt stocks, short-term (DOD, current...","1,305.87"
2,"External debt stocks, private nonguaranteed (P...",562.63
3,"External debt stocks, public and publicly guar...",504.11
4,"Debt service on external debt, total (TDS, cur...",352.50
5,"Debt service on external debt, public and publ...",79.27
6,"IBRD loans and IDA credits (DOD, current US$)",14.69
7,"PPG, IBRD (DOD, current US$)",14.69
8,"Multilateral debt service (TDS, current US$)",5.98
9,"PPG, IDA (DOD, current US$)",0.00


China's short-term external debt (due within a year) makes up more than half of its total
external debt stock — a structural feature worth flagging: short-term debt must be rolled over
constantly and is far more exposed to sudden shifts in investor confidence or global interest rates
than long-term public debt.


## Key findings

| Metric | Result |
|---|---|
| Countries covered | 120 |
| Debt indicators tracked | 10 |
| Combined external debt (all countries) | **$8.94 trillion** |
| Largest single debtor | **China** — $2,419.8B |
| Average debt per country | $75.7B |
| Highest debt-to-GNI ratio | **Mozambique** — 350.6% of GNI |
| Highest debt-service burden | **El Salvador** — 96.2% of export earnings |
| Only top-20 debtor with debt > GDP | **Ukraine** — 101.4% of GDP (war financing) |

See the [README](../README.md) for the full **Country Deep Dive** — causes, thriving industries,
political context, and debt-reduction levers for each of the top 10 most indebted and bottom 10
least indebted countries — and the [interactive dashboard](../assets/international_debt_dashboard.html)
for a visual, captioned walkthrough of every chart above.


In [17]:
conn.close()